<a href="https://colab.research.google.com/github/beriltezel/Evaluation-of-Text-Representation-Methods-for-ICONCLASS-Label-Recommendation/blob/main/Gemma_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U duckdb sentence-transformers transformers accelerate iconclass

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
import duckdb
import csv
import json
from datetime import datetime
from time import time

from google.colab import drive
from rich.progress import track
from iconclass import init
from sentence_transformers import SentenceTransformer
from huggingface_hub import login

In [ ]:
drive.mount('/content/drive')

In [ ]:
login("HF-TOKEN")

In [ ]:
USED_NOTATION_KEYS_PATH = "/content/drive/MyDrive/Colab Notebooks/used_notation_keys.txt"
GROUND_TRUTH_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/ground_truth.csv"
OUTPUT_JSONL_PATH = "/content/drive/MyDrive/Colab Notebooks/model_results.jsonl"

MODEL_LABEL = "gemma"
TOP_N = 20
LANG = "en"
BATCH_SIZE = 10000

In [ ]:
with open(USED_NOTATION_KEYS_PATH, "r", encoding="utf-8") as f:
    used_notation_keys = set(line.strip() for line in f if line.strip())

In [ ]:
DB = duckdb.connect("iconclass.duckdb")
DB.execute("INSTALL vss; LOAD vss;")
DB.execute("drop table if exists gemma")
DB.execute("create table gemma (notation TEXT, vec FLOAT[512])")

In [ ]:
model = SentenceTransformer("google/embeddinggemma-300m", truncate_dim=512, device="cuda")

In [ ]:
def do(batch, limit=BATCH_SIZE):
    if len(batch) < limit:
        return batch

    embeddings = model.encode_document(list(batch.values()))
    to_insert = zip(batch.keys(), embeddings)
    DB.executemany("INSERT INTO gemma VALUES (?, ?)", to_insert)
    DB.commit()
    return {}

In [ ]:
IC = init()
all = [x for x in IC.source._D.keys() if x is not None]

idx = 0
start_time = time()
batch = {}

for notation in track(all):
    if notation.find('(+') > 1 and notation not in used_notation_keys:
        continue

    n = IC[notation]

    buf = set([y for x in n.path() for y in x.keywords(LANG)])
    buf.add(n(LANG))

    batch[notation] = "\n".join(buf)
    batch = do(batch)

    if len(batch) == 0:
        print(f"At {idx}")

    idx += 1

do(batch, limit=0)

semantic_duration = time() - start_time
print(f"Gemma processing duration: {semantic_duration} seconds")
print(DB.execute("select count(*) from gemma").fetchall())

In [ ]:
def load_ground_truth(path, has_header=True):
    ground_truth = {}

    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f, delimiter=";")

        if has_header:
            next(reader, None)

        for row in reader:
            if not row:
                continue

            query = row[0].strip()

            if not query:
                continue

            relevant_codes = [
                cell.strip()
                for cell in row[1:]
                if cell and cell.strip()
            ]

            ground_truth[query] = relevant_codes

    return ground_truth

In [ ]:
def query_gemma(a, count=TOP_N):
    q = model.encode_query(a)

    results = []

    rows = DB.execute(
        """
        select notation,
               array_cosine_similarity(vec, ?::FLOAT[512]) as score
        from gemma
        order by score desc
        limit ?
        """,
        (q, count)
    ).fetchall()

    for rank, (notation, score) in enumerate(rows, start=1):
        n = IC[notation]

        try:
            label = n(LANG)
        except Exception:
            label = ""

        results.append({
            "rank": rank,
            "code": notation,
            "label": label,
            "score": float(score)
        })

    return results

In [ ]:
def append_jsonl(record, path=OUTPUT_JSONL_PATH):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
def check_ground_truth_against_corpus(ground_truth):
    valid_codes = set(row[0] for row in DB.execute("select notation from gemma").fetchall())
    missing = {}

    for query, gt_codes in ground_truth.items():
        missing_codes = [code for code in gt_codes if code not in valid_codes]

        if missing_codes:
            missing[query] = missing_codes

    return missing

In [ ]:
ground_truth = load_ground_truth(GROUND_TRUTH_CSV_PATH, has_header=True)
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
missing = check_ground_truth_against_corpus(ground_truth)

if missing:
    print("\nWarning: some ground-truth codes are not in the Gemma corpus.")
    print("These codes cannot be retrieved by Gemma under the shared filtering rule.")
    print(f"Queries affected: {len(missing)}\n")

for query, gt_codes in ground_truth.items():
    results = query_gemma(query, count=TOP_N)

    record = {
        "run_id": run_id,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "model": MODEL_LABEL,
        "model_name": "google/embeddinggemma-300m",
        "top_n": TOP_N,
        "query": query,
        "ground_truth_codes": gt_codes,
        "predicted_results": results,
        "predicted_codes": [item["code"] for item in results],
        "ground_truth_codes_missing_from_corpus": missing.get(query, [])
    }

    append_jsonl(record, path=OUTPUT_JSONL_PATH)

print("Finished Gemma run.")
print(f"Queries processed: {len(ground_truth)}")
print(f"Results appended to: {OUTPUT_JSONL_PATH}")